**Status: Battle & Event Predictions.** Overtake-probability model over `workspace.default.f1_cleaned_lap_dataset`: for pairs of drivers running close together, predicts whether the trailing driver passes the car ahead within the next few laps.

Reads the same source table as `phase_3_ranking.ipynb` but is otherwise independent — no dependency on that notebook having been run. Persists its model + circuit overtake-difficulty prior to the Unity Catalog Volume / tables also used by `phase_4.ipynb` and the Databricks App (`app/`).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve

In [0]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("F1-Battle-Model")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [0]:
MASTER_PATH = "workspace.default.f1_cleaned_lap_dataset"
df = spark.table(MASTER_PATH)

### Step 1: Per-driver lap state + gap-closing rate

In [0]:
N_LOOKAHEAD = 5
GAP_THRESHOLD = 3.0

driver_window = Window.partitionBy("Year", "Circuit", "Driver").orderBy("LapNumber")

pos_df = (
    df.select(
        "Year", "Circuit", "Driver", "LapNumber", "Position", "GapToAhead",
        "TyreLife", "Compound", "SpeedST"
    )
    .dropDuplicates(["Year", "Circuit", "Driver", "LapNumber"])
)

# future position, k laps ahead — used to label overtakes
for k in range(1, N_LOOKAHEAD + 1):
    pos_df = pos_df.withColumn(f"pos_plus_{k}", F.lead("Position", k).over(driver_window))

In [0]:
# gap-closing rate: least-squares slope of GapToAhead over the last 3 laps
closing_window = Window.partitionBy("Year", "Circuit", "Driver").orderBy("LapNumber").rowsBetween(-2, 0)

pos_df = (
    pos_df
    .withColumn("_gc_n", F.count("GapToAhead").over(closing_window))
    .withColumn("_gc_sum_x", F.sum("LapNumber").over(closing_window))
    .withColumn("_gc_sum_y", F.sum("GapToAhead").over(closing_window))
    .withColumn("_gc_sum_xy", F.sum(F.col("LapNumber") * F.col("GapToAhead")).over(closing_window))
    .withColumn("_gc_sum_xx", F.sum(F.col("LapNumber") * F.col("LapNumber")).over(closing_window))
)

_gc_denom = F.col("_gc_n") * F.col("_gc_sum_xx") - F.col("_gc_sum_x") * F.col("_gc_sum_x")

pos_df = pos_df.withColumn(
    "gap_closing_rate",
    F.when(
        (F.col("_gc_n") >= 2) & (_gc_denom != 0),
        (F.col("_gc_n") * F.col("_gc_sum_xy") - F.col("_gc_sum_x") * F.col("_gc_sum_y")) / _gc_denom
    ).otherwise(F.lit(None).cast("double"))
).drop("_gc_n", "_gc_sum_x", "_gc_sum_y", "_gc_sum_xy", "_gc_sum_xx")

### Step 2: Battle pairs + overtake label

In [0]:
# pair each driver with whoever is directly ahead, within GAP_THRESHOLD
trailing = pos_df.alias("t")
leading = pos_df.alias("l")

pairs = (
    trailing.join(
        leading,
        (F.col("t.Year") == F.col("l.Year"))
        & (F.col("t.Circuit") == F.col("l.Circuit"))
        & (F.col("t.LapNumber") == F.col("l.LapNumber"))
        & (F.col("t.Position") == F.col("l.Position") + 1),
    )
    .filter((F.col("t.GapToAhead") > 0) & (F.col("t.GapToAhead") <= GAP_THRESHOLD))
)

In [0]:
# label: does the trailing driver end up ahead within N_LOOKAHEAD laps
overtake_expr = F.lit(False)
for k in range(1, N_LOOKAHEAD + 1):
    overtake_expr = overtake_expr | (
        F.col(f"t.pos_plus_{k}").isNotNull()
        & F.col(f"l.pos_plus_{k}").isNotNull()
        & (F.col(f"t.pos_plus_{k}") < F.col(f"l.pos_plus_{k}"))
    )

battle_df = pairs.select(
    F.col("t.Year").alias("Year"),
    F.col("t.Circuit").alias("Circuit"),
    F.col("t.LapNumber").alias("LapNumber"),
    F.col("t.Driver").alias("TrailingDriver"),
    F.col("l.Driver").alias("LeadingDriver"),
    F.col("t.GapToAhead").alias("Gap"),
    F.col("t.gap_closing_rate").alias("GapClosingRate"),
    (F.col("t.TyreLife") - F.col("l.TyreLife")).alias("TyreLifeDelta"),
    F.col("t.Compound").alias("TrailingCompound"),
    F.col("l.Compound").alias("LeadingCompound"),
    (F.col("t.SpeedST") - F.col("l.SpeedST")).alias("SpeedST_Delta"),
    overtake_expr.cast("int").alias("Overtake"),
)

display(battle_df)

### Step 3: Circuit overtake-difficulty prior

In [0]:
# per (Circuit, Year), then an expanding prior using only PAST years —
# same leakage discipline as the driver/constructor profiles in phase_3_ranking.ipynb
circuit_year_stats = battle_df.groupBy("Circuit", "Year").agg(
    F.avg("Overtake").alias("YearOvertakeRate"),
    F.count("*").alias("YearBattleCount"),
)

circuit_expanding_window = (
    Window.partitionBy("Circuit").orderBy("Year").rowsBetween(Window.unboundedPreceding, -1)
)

circuit_prior_expanding = (
    circuit_year_stats
    .withColumn("_cum_battles", F.sum("YearBattleCount").over(circuit_expanding_window))
    .withColumn(
        "_cum_overtakes",
        F.sum(F.col("YearOvertakeRate") * F.col("YearBattleCount")).over(circuit_expanding_window),
    )
    .withColumn(
        "CircuitOvertakePrior",
        F.when(F.col("_cum_battles") > 0, F.col("_cum_overtakes") / F.col("_cum_battles")),
    )
    .select("Circuit", "Year", "CircuitOvertakePrior")
)

battle_df = battle_df.join(circuit_prior_expanding, ["Circuit", "Year"], "left")

In [0]:
# full-history version for inference (no future-row to leak once the model is deployed)
circuit_prior_full = battle_df.groupBy("Circuit").agg(
    F.avg("Overtake").alias("CircuitOvertakePrior"),
    F.count("*").alias("BattleCount"),
)

circuit_prior_full.write.mode("overwrite").saveAsTable("workspace.default.f1_circuit_overtake_prior")
display(circuit_prior_full)

### Step 4: Convert to pandas + train/val/test split

In [0]:
BATTLE_FEATURES = [
    "Gap", "GapClosingRate", "TyreLifeDelta",
    "TrailingCompound", "LeadingCompound",
    "SpeedST_Delta", "CircuitOvertakePrior",
]
BATTLE_CATEGORICAL = ["TrailingCompound", "LeadingCompound"]

pdf_battles = battle_df.select(BATTLE_FEATURES + ["Year", "Overtake"]).toPandas()

for c in BATTLE_CATEGORICAL:
    pdf_battles[c] = pdf_battles[c].astype("category")

pdf_battles.head()

In [0]:
TRAIN_END_YEAR = 2020
VAL_YEAR = 2021
TEST_START_YEAR = 2022

train_mask = pdf_battles["Year"] <= TRAIN_END_YEAR
val_mask = pdf_battles["Year"] == VAL_YEAR
test_mask = pdf_battles["Year"] >= TEST_START_YEAR

X_train = pdf_battles.loc[train_mask, BATTLE_FEATURES]
y_train = pdf_battles.loc[train_mask, "Overtake"]

X_val = pdf_battles.loc[val_mask, BATTLE_FEATURES]
y_val = pdf_battles.loc[val_mask, "Overtake"]

X_test = pdf_battles.loc[test_mask, BATTLE_FEATURES]
y_test = pdf_battles.loc[test_mask, "Overtake"]

print("Train/val/test rows:", len(X_train), len(X_val), len(X_test))
print("Overtake rate (train):", round(y_train.mean(), 4))

### Step 5: Train overtake-probability model

In [0]:
mlflow.set_registry_uri("databricks-uc")

battle_model_params = dict(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    early_stopping_rounds=30,
)

with mlflow.start_run(run_name="battle_overtake_model") as run:
    mlflow.log_params(battle_model_params)

    battle_model = XGBClassifier(**battle_model_params)
    battle_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    battle_probs_test = battle_model.predict_proba(X_test)[:, 1]

    battle_auc = roc_auc_score(y_test, battle_probs_test)
    battle_brier = brier_score_loss(y_test, battle_probs_test)
    battle_logloss = log_loss(y_test, battle_probs_test, labels=[0, 1])

    mlflow.log_metric("test_auc", battle_auc)
    mlflow.log_metric("test_brier", battle_brier)
    mlflow.log_metric("test_logloss", battle_logloss)

    battle_signature = infer_signature(X_train, battle_model.predict_proba(X_train))
    mlflow.xgboost.log_model(
        battle_model,
        "battle_model",
        signature=battle_signature,
        registered_model_name="workspace.default.f1_battle_model",
    )

print("Overtake model AUC:", round(battle_auc, 4))
print("Overtake model Brier score:", round(battle_brier, 4))
print("Overtake model log-loss:", round(battle_logloss, 4))

In [0]:
# reliability check — predict_proba is native here, no isotonic step needed
frac_pos, mean_pred = calibration_curve(y_test, battle_probs_test, n_bins=10, strategy="quantile")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.plot(mean_pred, frac_pos, marker="o", label="battle_model")
ax.set_xlabel("Predicted P(overtake within 5 laps)")
ax.set_ylabel("Observed overtake frequency")
ax.set_title("Battle model calibration (test seasons)")
ax.legend()
plt.show()

### What Plan G does *not* include yet

- **Track-limits / repeat-offender risk.** `session.race_control_messages` is now ingested by `phase_1.ipynb` into `workspace.default.f1_race_control_messages`, but this notebook doesn't yet build a driver-level risk feature from it or use it in `battle_model`. Left as a follow-up once a few backfill runs have populated the table.
- **Live wiring.** Validated the same way as Plan E (replay against finished races), not against an actual live feed — that needs Plan D, which isn't built.

### Step 6: Persist model + helper

In [0]:
STRATEGY_VOLUME_DIR = "/Volumes/workspace/default/f1_data"
BATTLE_VOLUME_DIR = f"{STRATEGY_VOLUME_DIR}/battle_models"

import os
os.makedirs(BATTLE_VOLUME_DIR, exist_ok=True)

battle_model.save_model(f"{BATTLE_VOLUME_DIR}/m_overtake_classifier.json")

In [0]:
def predict_overtake_probability(gap, gap_closing_rate, tyre_life_delta, trailing_compound, leading_compound, speedst_delta, circuit_overtake_prior):
    row = pd.DataFrame([{
        "Gap": gap,
        "GapClosingRate": gap_closing_rate,
        "TyreLifeDelta": tyre_life_delta,
        "TrailingCompound": trailing_compound,
        "LeadingCompound": leading_compound,
        "SpeedST_Delta": speedst_delta,
        "CircuitOvertakePrior": circuit_overtake_prior,
    }])
    for c in BATTLE_CATEGORICAL:
        row[c] = row[c].astype("category")
    return float(battle_model.predict_proba(row[BATTLE_FEATURES])[0, 1])

In [0]:
predict_overtake_probability(
    gap=0.8, gap_closing_rate=-0.3, tyre_life_delta=8,
    trailing_compound="SOFT", leading_compound="HARD",
    speedst_delta=4.0, circuit_overtake_prior=0.25,
)